# Family coverage diagnostic (Gamma `/markets` → rule hits)

Goal: sanity-check whether your **signal family rules** can find any matches in a broad, noisy sample of active markets.

If a family has **near-zero hits** even after sampling a few thousand markets, it's usually a **rule/keyword mismatch**.
If a family has hits here but **no tags** in tag-selection, it's usually a **discovery bias** (not seeing the right tags/events).


In [31]:
# Cell 0 — Imports + config

import os
from collections import defaultdict

import pandas as pd

from polyscanner.env import load_env
from polyscanner.clients.polymarket_gamma import fetch_and_normalize_active_markets
from polyscanner.pipeline.signal_family_mvp import match_market_to_rule
from polyscanner.signal_family_rules import RULES_BY_SLUG

load_env()

BASE_URL = os.getenv("POLYMARKET_API_BASE_URL") or "https://gamma-api.polymarket.com"

# How many markets to sample from Gamma /markets (increase if needed)
N_MARKETS_TARGET = 3000
PAGE_SIZE = 200

# Optional: focus inspection on a few families
FOCUS_FAMILIES = [
    "crypto_regime_changes",
    "antitrust_platforms_app_stores_ads",
    "us_china_semis_export_controls",
    "it_spending_cycle_enterprise_cloud_ai",
    "ai_regulation_big_tech_enforcement",
]


In [32]:
# Cell 1 — Fetch a broad sample of active markets from /markets

markets = []
offset = 0

while len(markets) < N_MARKETS_TARGET:
    batch = fetch_and_normalize_active_markets(
        base_url=BASE_URL,
        limit=PAGE_SIZE,
        offset=offset,
        timeout_s=30,
    )
    if not batch:
        break
    markets.extend(batch)
    offset += PAGE_SIZE

len(markets), markets[0].pm_market_id, markets[0].question[:80]


(3000, 517310, 'Will Trump deport less than 250,000?')

In [33]:
# Cell 2 — Build a DataFrame for exploration

df_m = pd.DataFrame(
    [
        {
            "pm_market_id": int(m.pm_market_id),
            "question": m.question,
            "category": m.category,
            "probability": m.probability,
            "volume_usd": m.volume,
        }
        for m in markets
    ]
)

df_m.shape, df_m.head(5)


((3000, 5),
    pm_market_id                                       question category  \
 0        517310           Will Trump deport less than 250,000?     None   
 1        517311      Will Trump deport 250,000-500,000 people?     None   
 2        517313     Will Trump deport 500,000-750,000- people?     None   
 3        517314    Will Trump deport 750,000-1,000,000 people?     None   
 4        517315  Will Trump deport 1,000,000-1,250,000 people?     None   
 
   probability    volume_usd  
 0        None  1.217521e+06  
 1        None  7.516491e+06  
 2        None  5.347826e+05  
 3        None  5.241418e+05  
 4        None  5.215140e+05  )

In [34]:
# Cell 3 — Run rule matching (market -> family)

rows = []
for _i, r in df_m.iterrows():
    q = str(r.get("question") or "")
    cat = str(r.get("category") or "")
    for slug, rule in RULES_BY_SLUG.items():
        score, terms = match_market_to_rule(question=q, category=cat, rule=rule)
        if score > 0:
            rows.append(
                {
                    "pm_market_id": int(r["pm_market_id"]),
                    "family": slug,
                    "match_score": float(score),
                    "matched_terms": terms,
                    "question": q,
                    "volume_usd": r.get("volume_usd"),
                }
            )

df_hits = pd.DataFrame(rows)
df_hits.shape, df_hits["pm_market_id"].nunique() if not df_hits.empty else 0


((81, 6), 81)

In [35]:
# Cell 4 — Coverage: how many unique markets match each family?

if df_hits.empty:
    print("No matches found at all. Rules are likely too strict or market text fields changed.")
else:
    display(
        df_hits.groupby("family")["pm_market_id"].nunique().sort_values(ascending=False)
    )


family
fomc_surprises                       46
crypto_regime_changes                26
datacenter_power_grid_constraints     6
taiwan_geopolitical_risk              3
Name: pm_market_id, dtype: int64

In [36]:
# Cell 5 — Inspect top matches for a given family

FAMILY = "crypto_regime_changes"  # change me

if df_hits.empty:
    print("df_hits is empty")
else:
    d = df_hits[df_hits["family"] == FAMILY].copy()
    d["volume_usd"] = pd.to_numeric(d["volume_usd"], errors="coerce")
    display(
        d.sort_values(["match_score", "volume_usd"], ascending=False)
        .head(20)[["pm_market_id", "match_score", "matched_terms", "volume_usd", "question"]]
    )


,pm_market_id,match_score,matched_terms,volume_usd,question
61,659035,0.142857,[sec],22424.280666,Will Missouri (Women’s) win the 2025–2026 SEC ...
70,662906,0.142857,[sec],16507.776487,Will Florida win the 2025-2026 SEC Men's Baske...
71,662908,0.142857,[sec],1172.630000,Will Kentucky win the 2025-2026 SEC Men's Bask...
76,662918,0.142857,[sec],854.266619,Will Vanderbilt win the 2025-2026 SEC Men's Ba...
77,662932,0.142857,[sec],829.306000,Will another team win the 2025-2026 SEC Men's ...
69,662904,0.142857,[sec],673.504039,Will Arkansas win the 2025-2026 SEC Men's Bask...
74,662916,0.142857,[sec],670.291234,Will Texas A&M win the 2025-2026 SEC Men's Bas...
72,662911,0.142857,[sec],646.300000,Will Missouri win the 2025-2026 SEC Men's Bask...
68,662903,0.142857,[sec],638.300000,Will Alabama win the 2025-2026 SEC Men's Baske...
75,662917,0.142857,[sec],563.300000,Will Texas win the 2025-2026 SEC Men's Basketb...


In [37]:
# Cell 6 — If a family has 0 hits, search for likely phrases in raw market questions

def search_questions(df: pd.DataFrame, phrases: list[str], *, max_rows: int = 25) -> pd.DataFrame:
    """Return markets whose question contains any of the provided phrases.

    Notes:
    - Uses simple substring matching (regex=False) for transparency.
    - If `phrases` is empty, returns an empty DataFrame (prevents `KeyError: False`).
    """
    cols = ["pm_market_id", "question", "category", "volume_usd"]
    if not phrases:
        return df.iloc[0:0][cols]
    s = df["question"].fillna("").astype(str).str.lower()
    mask = pd.Series(False, index=df.index)
    for p in phrases:
        mask = mask | s.str.contains(str(p).lower(), regex=False)
    return df.loc[mask, cols].head(int(max_rows))


# Tune per family:
SEARCH_HINTS = {
    "crypto_regime_changes": [
        "stablecoin",
        # "sec",
        "cftc",
        "etf",
        "gensler",
        "exchange",
        "coinbase",
    ],
    # Avoid over-broad substrings like "ad" (matches "Madrid").
    "antitrust_platforms_app_stores_ads": [
        "antitrust",
        "ftc",
        "doj",
        "app store",
        "sideload",
        "sideloading",
        "digital markets act",
        "dma",
        "advertising",
    ],
    # Keep these specific; adding "china" floods results.
    "us_china_semis_export_controls": [
        "export controls",
        "chip ban",
        "semiconductor",
        "asml",
        "tsmc",
        "nvidia",
        "h20",
    ],
    "it_spending_cycle_enterprise_cloud_ai": [
        "aws",
        "azure",
        "gcp",
        "cloud",
        "saas",
        "capex",
        "datacenter",
        "data center",
    ],
    "ai_regulation_big_tech_enforcement": [
        "antitrust",
        "doj",
        "ftc",
        "ai act",
        "ai regulation",
        "platform regulation",
    ],
}

for fam in FOCUS_FAMILIES:
    print("\n==", fam, "==")
    if not df_hits.empty and (df_hits[df_hits["family"] == fam]["pm_market_id"].nunique() > 0):
        print("Has rule hits; inspect with Cell 5.")
        continue
    hints = SEARCH_HINTS.get(fam, [])
    display(search_questions(df_m, hints, max_rows=30))



== crypto_regime_changes ==
Has rule hits; inspect with Cell 5.

== antitrust_platforms_app_stores_ads ==


,pm_market_id,question,category,volume_usd
2322,645181,Will Victor Hedman win the 2025–2026 NHL Art R...,None,1719.7020
2406,645340,Will Victor Hedman win the 2025–2026 NHL Hart ...,None,2450.4503
2598,645616,Will Victor Hedman win the 2025–2026 NHL James...,None,474.7167



== us_china_semis_export_controls ==


,pm_market_id,question,category,volume_usd
1578,631181,Will NVIDIA be the largest company in the worl...,None,343129.885409



== it_spending_cycle_enterprise_cloud_ai ==


,pm_market_id,question,category,volume_usd



== ai_regulation_big_tech_enforcement ==


,pm_market_id,question,category,volume_usd
